In [ ]:
from dataclasses import dataclass
import math
import random
import collections
import numpy as np
import os
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS
from collections import Counter
import tqdm
import re
import torch
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
import time
from datetime import datetime
import exiftool
import enum


In [ ]:
DATA_ROOT = '/home/slavik/e202602_eclipse/data'
BRIGHTNESS_MIN = 0.0
BRIGHTNESS_MAX = 1.0
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
class MoonInfoOrigin(enum.Enum):
    DIRECT = 0
    INTERPOLATED = 1

In [ ]:

@dataclass
class ImageInfo:
    path: Path
    width: int
    height: int
    avg_brightness: float
    timestamp: float
    exposure_time: float
    moon: tuple[float, float, float] = None    # (center_i, center_j, radius) in pixels
    moon_info_origin: MoonInfoOrigin = None
    moon_pos_std_px: float = None



In [ ]:
def get_info_from_exif(img_path: Path) -> float:
    """
    Extract exposure time in seconds from image EXIF via exiftool.
    Raises ValueError if EXIF:ExposureTime is missing.
    """
    with exiftool.ExifToolHelper() as et:
        metadata = et.get_metadata(str(img_path))[0]
    exposure_time = metadata.get('EXIF:ExposureTime')
    assert exposure_time is not None
    assert isinstance(exposure_time, (int, float)), type(exposure_time)
    # Composite:SubSecDateTimeOriginal 2024:04:08 15:27:05.89-04:00
    subsec_date_time_original = metadata.get('Composite:SubSecDateTimeOriginal')
    assert subsec_date_time_original is not None
    assert re.match(r'\d{4}:\d{2}:\d{2} \d{2}:\d{2}:\d{2}\.\d{2}-\d{2}:\d{2}', subsec_date_time_original), subsec_date_time_original
    expected_format = "%Y:%m:%d %H:%M:%S.%f%z"
    try:
        dt = datetime.strptime(subsec_date_time_original, expected_format)
        timestamp = dt.timestamp()
    except ValueError as e:
        raise ValueError(f"Failed to parse DateTime '{subsec_date_time_original}' in {img_path}: {e}")
    return float(exposure_time), timestamp


def get_image_infos():
    jpg_files = list(Path(DATA_ROOT).rglob('*.jpg')) + list(Path(DATA_ROOT).rglob('*.JPG'))
    image_infos = []
    for jpg_file in tqdm.tqdm(jpg_files, desc="First scan of images"):
        with Image.open(jpg_file) as img:
            width, height = img.size
            avg_brightness = np.array(img).astype(np.float32).mean() / 255.0
            if BRIGHTNESS_MIN <= avg_brightness <= BRIGHTNESS_MAX:
                exposure_time, timestamp = get_info_from_exif(jpg_file)
                image_infos.append(ImageInfo(path=jpg_file, width=width, height=height, avg_brightness=avg_brightness, timestamp=timestamp, exposure_time=exposure_time))
    assert len(image_infos) > 0
    for ii in image_infos:
        assert ii.width == image_infos[0].width
        assert ii.height == image_infos[0].height
    image_infos.sort(key=lambda x: x.avg_brightness)
    return image_infos

In [ ]:
N_SECTORS = 360
N_TRIPLETS = 1024
N_CLUSTER = 256
DEBUG_RADIUS_PX = 4
REFINE_ITERATIONS = 3
MIN_TRIPLET_DEGREES = 30


def _indices_within_degrees(center: int, deg: int) -> set:
    """Sector indices within deg degrees of center (wrap-around)."""
    return {(center + d) % N_SECTORS for d in range(-(deg - 1), deg)}


def sample_triplet_indices(n_pts: int, min_degrees: int = MIN_TRIPLET_DEGREES) -> tuple[int, int, int]:
    """
    Sample three distinct sector indices such that each pair is at least min_degrees apart.
    Falls back to unrestricted random triplet if not enough spread is available.
    """
    available = set(range(n_pts))
    a = random.sample(list(available), 1)[0]
    available -= _indices_within_degrees(a, min_degrees) & available
    assert len(available) >= 2
    b = random.sample(list(available), 1)[0]
    available -= _indices_within_degrees(b, min_degrees) & available
    assert len(available) >= 1
    c = random.sample(list(available), 1)[0]
    return (a, b, c)


def refine_moon(img: torch.Tensor, center_i: float, center_j: float) -> tuple[float, float, float]:
    """
    Refine moon center from image and current center (i, j).
    img: (H, W, 3) float32 [0,1] on GPU.
    Returns (i, j, radius) as tuple of (float, float, float).
    """
    assert img.ndim == 3 and img.shape[2] == 3
    H, W = img.shape[0], img.shape[1]
    dev = img.device
    img_size = float(max(H, W))

    gray = img.mean(dim=2)
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    g = gray.unsqueeze(0).unsqueeze(0)
    grad_x = torch.nn.functional.conv2d(g, sobel_x, padding=1).squeeze()
    grad_y = torch.nn.functional.conv2d(g, sobel_y, padding=1).squeeze()

    dy = torch.arange(H, device=dev, dtype=torch.float32).view(-1, 1) - center_i
    dx = torch.arange(W, device=dev, dtype=torch.float32).view(1, -1) - center_j
    norm = torch.sqrt(dx * dx + dy * dy).clamp(min=1e-6)
    u_x = dx / norm
    u_y = dy / norm

    angle = torch.atan2(dy, dx)
    sector_id = (torch.floor((angle + math.pi) / (2 * math.pi) * N_SECTORS).long() % N_SECTORS)

    dot_product = grad_x * u_x + grad_y * u_y
    dot_product_flat = dot_product.reshape(-1)
    sector_flat = sector_id.reshape(-1)
    W_t = W

    points_list = []
    for s in range(N_SECTORS):
        mask = sector_flat == s
        if mask.any():
            masked = torch.where(mask, dot_product_flat, torch.tensor(-1e9, device=dev, dtype=torch.float32))
            idx = masked.argmax().item()
            i, j = idx // W_t, idx % W_t
            points_list.append((i, j))

    n_pts = len(points_list)
    if n_pts < 3:
        return (center_i, center_j, 0.0)

    def circumcenter(i1, j1, i2, j2, i3, j3):
        x1, y1, x2, y2, x3, y3 = float(j1), float(i1), float(j2), float(i2), float(j3), float(i3)
        D = 2.0 * (x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
        if abs(D) < 1e-10:
            return None
        ox = ((x1 * x1 + y1 * y1) * (y2 - y3) + (x2 * x2 + y2 * y2) * (y3 - y1) + (x3 * x3 + y3 * y3) * (y1 - y2)) / D
        oy = ((x1 * x1 + y1 * y1) * (x3 - x2) + (x2 * x2 + y2 * y2) * (x1 - x3) + (x3 * x3 + y3 * y3) * (x2 - x1)) / D
        oi, oj = oy, ox
        d1 = math.hypot(i1 - oi, j1 - oj)
        d2 = math.hypot(i2 - oi, j2 - oj)
        d3 = math.hypot(i3 - oi, j3 - oj)
        if d1 > img_size or d2 > img_size or d3 > img_size:
            return None
        # Compute radius as average of distances from circumcenter to the 3 points
        radius = (d1 + d2 + d3) / 3.0
        return (oi, oj, radius)

    circumcenters = []
    radii = []
    while len(circumcenters) < N_TRIPLETS:
        a, b, c = sample_triplet_indices(n_pts)
        i1, j1 = points_list[a]
        i2, j2 = points_list[b]
        i3, j3 = points_list[c]
        cc_result = circumcenter(i1, j1, i2, j2, i3, j3)
        if cc_result is not None:
            oi, oj, radius = cc_result
            circumcenters.append((oi, oj))
            radii.append(radius)

    pts = np.array(circumcenters, dtype=np.float64)
    Z = linkage(pts, method="complete")
    t_lo, t_hi = 0.0, float(Z[-1, 2])
    for _ in range(60):
        t = (t_lo + t_hi) / 2
        labels = fcluster(Z, t, criterion="distance")
        sizes = np.bincount(labels)
        max_size = int(sizes.max())
        if max_size >= N_CLUSTER:
            t_hi = t
        else:
            t_lo = t
    labels = fcluster(Z, t_hi, criterion="distance")
    sizes = np.bincount(labels)
    which = int(np.argmax(sizes))
    cluster_mask = labels == which
    cluster_pts = pts[cluster_mask]
    cluster_radii = np.array(radii)[cluster_mask]
    ci = float(cluster_pts[:, 0].mean())
    cj = float(cluster_pts[:, 1].mean())
    radius = float(cluster_radii.mean())
    return (ci, cj, radius)


def find_moon(img: torch.Tensor, i0: float, j0: float) -> tuple[float, float, float]:
    """
    Find moon center by iteratively refining from image center.
    img: (H, W, 3) float32 [0,1] on GPU. Returns (i, j, radius) as tuple of (float, float, float).
    """
    assert img.ndim == 3 and img.shape[2] == 3
    center_i, center_j = i0, j0
    radius = 0.0
    for _ in range(REFINE_ITERATIONS):
        center_i, center_j, radius = refine_moon(img, center_i, center_j)

    if False:
        img_np = img.cpu().numpy()
        H, W = img_np.shape[0], img_np.shape[1]
        
        # Calculate crop size: 2.2 * radius (10% padding on each side)
        half_crop = int(round(1.1 * radius))
        
        # Calculate crop bounds (square crop centered on moon)
        i_min = int(round(center_i - half_crop))
        i_max = int(round(center_i + half_crop))
        j_min = int(round(center_j - half_crop))
        j_max = int(round(center_j + half_crop))
        
        # Check bounds and throw exception if out of bounds
        if i_min < 0 or i_max >= H or j_min < 0 or j_max >= W:
            raise ValueError(f"Crop bounds out of image: half_crop={half_crop}, center=({center_i}, {center_j}), radius={radius:.1f}, image_size=({H}, {W}), bounds=({i_min}, {i_max}, {j_min}, {j_max})")
        
        # Crop image
        img_cropped = img_np[i_min:i_max+1, j_min:j_max+1]
        
        # Create figure
        plt.figure(figsize=(12, 12))
        plt.imshow(img_cropped)
        
        # Draw center as small green circle (relative to cropped image)
        center_j_crop = center_j - j_min
        center_i_crop = center_i - i_min
        plt.gca().add_patch(plt.Circle((center_j_crop, center_i_crop), DEBUG_RADIUS_PX, color="green", fill=True))
        
        # Draw 36 equally spaced green pixels on the circle border
        n_points = 36
        for k in range(n_points):
            angle = 2 * math.pi * k / n_points
            border_j = center_j_crop + radius * math.cos(angle)
            border_i = center_i_crop + radius * math.sin(angle)
            border_j_int = int(round(border_j))
            border_i_int = int(round(border_i))
            # Draw single green pixel
            plt.plot(border_j_int, border_i_int, 'g.', markersize=1)
        
        plt.title(f"Moon center: (i={center_i}, j={center_j}), radius: {radius:.1f}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    return (center_i, center_j, radius)


class ApproxMoonFinder:
    """
    Approximate moon finder using circle edge detection.
    Precomputes circle kernels for efficient processing.
    """
    # Class attributes for precomputed kernels
    _kernels = {}  # Dict mapping radius -> kernel tensor
    _min_radius = 3
    _target_size = 256
    _max_radius = _target_size // 2 - 3
    
    @classmethod
    def _create_circle_kernel(cls, radius: int, device: torch.device) -> torch.Tensor:
        """
        Create a circle kernel with 1px thick border using distance-based approach.
        Returns kernel of shape (1, 1, kernel_size, kernel_size) on specified device.
        """
        kernel_size = 2 * radius + 1
        center = radius
        
        # Create coordinate grids
        y = torch.arange(kernel_size, dtype=torch.float32, device=device)
        x = torch.arange(kernel_size, dtype=torch.float32, device=device)
        yy, xx = torch.meshgrid(y, x, indexing='ij')
        
        # Compute distance from center
        dist = torch.sqrt((yy - center) ** 2 + (xx - center) ** 2)
        
        # Set to 1 if distance is within 0.5 of radius (1px thick border)
        kernel = (torch.abs(dist - radius) < 0.5).float()
        
        # Reshape for conv2d: (1, 1, H, W)
        return kernel.unsqueeze(0).unsqueeze(0)
    
    @classmethod
    def _get_kernel(cls, radius: int, device: torch.device) -> torch.Tensor:
        """Get or create kernel for given radius."""
        if radius not in cls._kernels:
            cls._kernels[radius] = cls._create_circle_kernel(radius, device)
        # Move kernel to requested device if needed
        kernel = cls._kernels[radius]
        if kernel.device != device:
            kernel = kernel.to(device)
            cls._kernels[radius] = kernel
        return kernel
    
    @classmethod
    def find_moon_approx(cls, img: torch.Tensor) -> tuple[int, int]:
        """
        Find moon center using approximate circle edge detection.
        img: (H, W, 3) float32 [0,1] on GPU. Returns (i, j) as tuple of ints in original image space.
        """
        assert img.ndim == 3 and img.shape[2] == 3
        original_H, original_W = img.shape[0], img.shape[1]
        device = img.device
        
        # Grayscale and downscale preserving aspect ratio, then pad/crop to 512x512
        gray = img.mean(dim=2)  # (H, W)
        
        # Compute downscaled dimensions preserving aspect ratio
        # Scale factor is min(target_size / original_size) to fit within target_size
        scale = min(cls._target_size / original_H, cls._target_size / original_W)
        H_scaled = int(round(original_H * scale))
        W_scaled = int(round(original_W * scale))
        
        # Downscale preserving aspect ratio
        gray_4d = gray.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W) for interpolation
        gray_scaled = torch.nn.functional.interpolate(
            gray_4d, size=(H_scaled, W_scaled), 
            mode='bilinear', align_corners=False
        ).squeeze()  # (H_scaled, W_scaled)
        
        # Pad to 512x512 (centered)
        pad_h = (cls._target_size - H_scaled) // 2
        pad_w = (cls._target_size - W_scaled) // 2
        gray_downscaled = torch.nn.functional.pad(
            gray_scaled, 
            (pad_w, cls._target_size - W_scaled - pad_w, pad_h, cls._target_size - H_scaled - pad_h),
            mode='constant', value=0.0
        )  # (512, 512)
        
        H_down, W_down = gray_downscaled.shape
        assert H_down == cls._target_size and W_down == cls._target_size
        
        # Initialize tracking variables
        best_diff = torch.full((H_down, W_down), float('-inf'), device=device, dtype=torch.float32)
        sum_prev = None  # Sum from 1 iteration ago
        sum_prev2 = None  # Sum from 2 iterations ago
        
        # Iterate over radii from min_radius to max_radius
        for radius in range(cls._min_radius, cls._max_radius + 1):
            # Get kernel for this radius
            kernel = cls._get_kernel(radius, device)
            
            # Each kernel needs padding equal to its radius to produce 512x512 output
            # This ensures: output_size = 512 + 2*radius - (2*radius+1) + 1 = 512
            # and all outputs are properly aligned (each pixel corresponds to same input location)
            padding = radius
            
            # Compute sum along circle using conv2d with per-kernel padding
            gray_input = gray_downscaled.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
            sum_along_circle = torch.nn.functional.conv2d(
                gray_input, kernel, padding=padding
            ).squeeze()  # (H, W)
            
            # Verify output size is correct (should always be 512x512)
            assert sum_along_circle.shape == (H_down, W_down), \
                f"Output shape mismatch: expected ({H_down}, {W_down}), got {sum_along_circle.shape} for radius {radius}"
            
            # If we have sum from 2 iterations ago, compute diff
            if sum_prev2 is not None:
                # diff = sum_now - sum_2iter_before_now
                diff = sum_along_circle - sum_prev2
                # Update best diff per pixel
                best_diff = torch.maximum(best_diff, diff)
            
            # Update history: shift by one iteration
            sum_prev2 = sum_prev
            sum_prev = sum_along_circle
        
        # Find pixel with maximum diff
        flat_idx = best_diff.argmax().item()
        i_down = flat_idx // W_down
        j_down = flat_idx % W_down
        
        # Map coordinates back to original image space
        # First, subtract padding offsets to get coordinates in scaled (non-padded) space
        i_scaled = i_down - pad_h
        j_scaled = j_down - pad_w
        
        # Then scale back to original image space
        # With align_corners=False, the mapping uses half-pixel alignment:
        # output_pos = (input_pos + 0.5) * (output_size / input_size) - 0.5
        # Inverse: input_pos = (output_pos + 0.5) * (input_size / output_size) - 0.5
        i = (i_scaled + 0.5) * (original_H / H_scaled) - 0.5
        j = (j_scaled + 0.5) * (original_W / W_scaled) - 0.5
        i = int(round(i))
        j = int(round(j))
        
        if False:
            # Visualize result
            img_np = img.cpu().numpy()
            plt.figure(figsize=(12, 8))
            plt.imshow(img_np)
            plt.gca().add_patch(plt.Circle((j, i), DEBUG_RADIUS_PX, color="green", fill=True))
            plt.title(f"Moon center (approx): (i={i}, j={j})")
            plt.axis("off")
            plt.tight_layout()
            plt.show()
        
        return (i, j)
    

In [ ]:
image_infos = get_image_infos()

for ii in tqdm.tqdm(image_infos, desc="Finding moon"):
    img = Image.open(ii.path)
    img_arr = torch.from_numpy(np.array(img).astype(np.float32) / 255.0).cuda()
    i0, j0 = ApproxMoonFinder.find_moon_approx(img_arr)
    i, j, radius = find_moon(img_arr, i0, j0)
    ii.moon = (i, j, radius)
    ii.moon_info_origin = MoonInfoOrigin.DIRECT

In [ ]:
exposure_groups = collections.defaultdict(list)
for ii in image_infos:
    exposure_groups[ii.exposure_time].append(ii)
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    radii = [ii.moon[2] for ii in group]
    brightnesses = [ii.avg_brightness for ii in group]
    print(f'{exposure_time=:.5f} {len(group)=} {np.mean(radii)=:.2f} {np.std(radii)=:.2f} {np.mean(brightnesses)=:.6f} {np.std(brightnesses)=:.8f}')


In [ ]:
RADIUS_STD_THRESHOLD = 1.0
MIN_GROUP_ELMS = 3

def radius(ii):
    return ii.moon[2]

prev_avg_radius = None
sorted_exposure_times = sorted(exposure_groups.keys())

for exposure_time in sorted_exposure_times:
    group = list(exposure_groups[exposure_time])
    assert len(group) >= MIN_GROUP_ELMS
    radii = [radius(ii) for ii in group]
    mean_r = np.mean(radii)
    std_r = np.std(radii)
    mean_r_ok = prev_avg_radius is None or abs(mean_r - prev_avg_radius) <= 0.1 * prev_avg_radius
    if std_r <= RADIUS_STD_THRESHOLD and mean_r_ok:
        prev_avg_radius = mean_r
    else:
        assert prev_avg_radius is not None, 'Failure in first group, it was expected to work allways'
        subgroup = [ii for ii in group if abs(radius(ii) - prev_avg_radius) <= 0.1 * prev_avg_radius]
        if len(subgroup) < MIN_GROUP_ELMS:
            subgroup = []
        else:
            mean_r = np.mean([radius(ii) for ii in subgroup])
            std_r = np.std([radius(ii) for ii in subgroup])
            if std_r > RADIUS_STD_THRESHOLD:
                subgroup = []
            else:
                prev_avg_radius = mean_r
        for ii in group:
            if ii not in subgroup:
                ii.moon = None
                ii.moon_info_origin = None

In [ ]:
# do linear interpolation: fill moon (i, j, radius) for image_infos where moon is None,
# using linear (i, j) vs time and radius from group average or last group with moons.

# 1. Build linear model (i, j) = f(timestamp) from image_infos that have moon is not None
pts_with_moon = [(ii.timestamp, ii.moon[0], ii.moon[1]) for ii in image_infos if ii.moon is not None]
assert len(pts_with_moon) >= 2, "Need at least 2 points with moon to fit linear model"
t_arr = np.array([p[0] for p in pts_with_moon], dtype=np.float64)
i_arr = np.array([p[1] for p in pts_with_moon], dtype=np.float64)
j_arr = np.array([p[2] for p in pts_with_moon], dtype=np.float64)
# i = a_i * t + b_i, j = a_j * t + b_j
(a_i, b_i) = np.polyfit(t_arr, i_arr, 1)
(a_j, b_j) = np.polyfit(t_arr, j_arr, 1)

def interpolate_moon_at_time(t: float) -> tuple[float, float]:
    return (float(a_i * t + b_i), float(a_j * t + b_j))

# 2. Iterate exposure groups in ascending exposure time; fill None moons and track radius
last_avg_radius = None
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    with_moon = [ii for ii in group if ii.moon is not None]
    if with_moon:
        avg_radius = float(np.mean([ii.moon[2] for ii in with_moon]))
        last_avg_radius = avg_radius
    else:
        assert last_avg_radius is not None
        avg_radius = last_avg_radius
    for ii in group:
        if ii.moon is None:
            i_pred, j_pred = interpolate_moon_at_time(ii.timestamp)
            ii.moon = (i_pred, j_pred, avg_radius)
            ii.moon_info_origin = MoonInfoOrigin.INTERPOLATED

In [ ]:
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    radii = [ii.moon[2] for ii in group]
    brightnesses = [ii.avg_brightness for ii in group]
    print(f'{exposure_time=:.5f} {len(group)=} {np.mean(radii)=:.2f} {np.std(radii)=:.2f} {np.mean(brightnesses)=:.6f} {np.std(brightnesses)=:.8f}')

In [ ]:
# estimate uncertainty of moon position: residual of DIRECT positions vs linear model, then max of std_i, std_j
residuals_i = []
residuals_j = []
for ii in image_infos:
    if ii.moon_info_origin != MoonInfoOrigin.DIRECT:
        continue
    i_pred, j_pred = interpolate_moon_at_time(ii.timestamp)
    residuals_i.append(ii.moon[0] - i_pred)
    residuals_j.append(ii.moon[1] - j_pred)
residuals_i = np.array(residuals_i)
residuals_j = np.array(residuals_j)
std_i = float(np.std(residuals_i))
std_j = float(np.std(residuals_j))
moon_position_uncertainty_px = max(std_i, std_j)
print(f'moon_position_uncertainty_px={moon_position_uncertainty_px:.2f}')

In [ ]:
# Add moon_pos_std_px to each image_info: per-group std of DIRECT radii when enough DIRECT, else moon_position_uncertainty_px
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    direct_in_group = [ii for ii in group if ii.moon_info_origin == MoonInfoOrigin.DIRECT]
    std_direct = None
    if len(direct_in_group) >= MIN_GROUP_ELMS:
        std_direct = float(np.std([ii.moon[2] for ii in direct_in_group]))
    for ii in group:
        if ii.moon_info_origin == MoonInfoOrigin.DIRECT and std_direct is not None:
            ii.moon_pos_std_px = std_direct
        else:
            ii.moon_pos_std_px = moon_position_uncertainty_px

In [ ]:
# Bad news about sun / moon apparent motion
# https://chatgpt.com/share/e/6996a405-7df4-800e-9c16-51b51a28ce9e

# if solar radius is 500px, then:
# 1px ~ 1.92 arcsec
# scene as a whole will move by 2344 px in 5 minutes (becays its 15deg in hour, so in pixels and in 5min we have (15*3600/1.92) * (5/60))
# moon apparent movement wrt sun: 80px in 5min
# moon radius minus sun radius: maximally 40px

In [ ]:
def register_equal_exposure(image_info0, image_info1):
    """
    Find (shift_i, shift_j, rotation_deg) to align the second image to the first.
    rotation_deg: degrees counterclockwise. shifts in pixels.
    Uses iterative 5x5x5 grid search on GPU; grid can reduce to 5x5x1 or 1x1x5 when one dimension converges.
    """
    device = torch.device("cuda")
    # Load images: grayscale float32 [0,1] on GPU
    def load_grayscale(ii):
        with Image.open(ii.path) as img:
            arr = np.array(img).astype(np.float32) / 255.0
        if arr.ndim == 3:
            arr = arr.mean(axis=2)
        return torch.from_numpy(arr).to(device=device, dtype=torch.float32)

    g0 = load_grayscale(image_info0)   # (H, W)
    g1 = load_grayscale(image_info1)   # (H, W)
    H, W = g0.shape
    assert g1.shape == (H, W)

    # Moon data (center_i, center_j, radius)
    moon0 = image_info0.moon
    moon1 = image_info1.moon
    r0 = moon0[2]
    r1 = moon1[2]
    moon_radius_avg = (r0 + r1) / 2.0

    # Uncertainty and sun drift
    u0 = image_info0.moon_pos_std_px if image_info0.moon_pos_std_px is not None else 2.0
    u1 = image_info1.moon_pos_std_px if image_info1.moon_pos_std_px is not None else 2.0
    sun_drift_per_sec = 0.001 * moon_radius_avg
    dt_sec = abs(image_info1.timestamp - image_info0.timestamp)
    possible_sun_drift = dt_sec * sun_drift_per_sec
    initial_shift_half = 5.0 * (u0 + u1) + possible_sun_drift
    # double it to stay safe
    initial_shift_half = 2.0 * (initial_shift_half + 3.0)

    # Rotation center = image center
    ci, cj = H / 2.0, W / 2.0
    r_border = max(H, W) # max distance from center to corner (in fact its too much, but it works)

    def fill_moon_circle(g, moon_center_i, moon_center_j, fill_radius):
        y = torch.arange(H, device=device, dtype=torch.float32).view(-1, 1) - moon_center_i
        x = torch.arange(W, device=device, dtype=torch.float32).view(1, -1) - moon_center_j
        r = torch.sqrt(y * y + x * x)
        return torch.where(r <= fill_radius, torch.ones_like(g, device=device), g)

    g0_filled = fill_moon_circle(g0.clone(), moon0[0], moon0[1], r0 + 3.0)
    g1_filled = fill_moon_circle(g1.clone(), moon1[0], moon1[1], r1 + 3.0)

    # Precompute output pixel coords (1, H, W) for batched broadcast with (N, 1, 1)
    ii = torch.arange(H, device=device, dtype=torch.float32).view(-1, 1).expand(H, W).unsqueeze(0)
    jj = torch.arange(W, device=device, dtype=torch.float32).view(1, -1).expand(H, W).unsqueeze(0)

    def apply_transform_batched(img, shift_i_t, shift_j_t, cos_a_t, sin_a_t):
        # shift_* (N,1,1), cos_a_t, sin_a_t (N,1,1). Output (N, H, W).
        di = ii - ci - shift_i_t   # (N, H, W)
        dj = jj - cj - shift_j_t
        i_src = di * cos_a_t + dj * sin_a_t + ci
        j_src = -di * sin_a_t + dj * cos_a_t + cj
        j_norm = 2.0 * j_src / (W - 1) - 1.0 if W > 1 else torch.zeros_like(j_src)
        i_norm = 2.0 * i_src / (H - 1) - 1.0 if H > 1 else torch.zeros_like(i_src)
        grid = torch.stack([j_norm, i_norm], dim=-1)  # (N, H, W, 2)
        img_4d = img.unsqueeze(0).unsqueeze(1)  # (1, 1, H, W)
        img_4d = img_4d.expand(grid.shape[0], 1, H, W)
        out = torch.nn.functional.grid_sample(
            img_4d, grid, mode="bilinear", padding_mode="zeros", align_corners=True
        )
        return out.squeeze(1)  # (N, H, W)

    def discrepancy_batched(target, warped):
        # target (H,W), warped (N, H, W). Return (N,) errors.
        both_lt_08 = (target < 0.8) & (warped < 0.8)
        any_gt_0 = (target > 0) | (warped > 0)
        mask = both_lt_08 & any_gt_0
        n = mask.sum(dim=(1, 2)).float().clamp(min=1.0)
        return (torch.abs(target - warped) * mask.float()).sum(dim=(1, 2)) / n

    # Grid search state
    best_shift_i = 0.0
    best_shift_j = 0.0
    best_angle = 0.0
    step_shift = initial_shift_half / 2.0   # 5 points over ±initial_shift_half
    step_angle = 5.0   # 5 points from -10 to +10 deg
    refine_shift = True
    refine_angle = True

    while refine_shift or refine_angle:
        if step_shift < 0.1:
            refine_shift = False
        if step_angle * r_border * (math.pi / 180.0) < 0.1:
            refine_angle = False

        shift_i_vals = (
            [best_shift_i] if not refine_shift
            else [best_shift_i + step_shift * (k - 2) for k in range(5)]
        )
        shift_j_vals = (
            [best_shift_j] if not refine_shift
            else [best_shift_j + step_shift * (k - 2) for k in range(5)]
        )
        angle_vals = (
            [best_angle] if not refine_angle
            else [best_angle + step_angle * (k - 2) for k in range(5)]
        )

        triples = [(si, sj, a) for si in shift_i_vals for sj in shift_j_vals for a in angle_vals]
        # Process in small batches to avoid OOM (full grid would be 125 x H x W)
        GRID_BATCH_SIZE = 8  # decrease to 2 or 1 if still OOM on large images
        best_err = float("inf")
        best_triple = (best_shift_i, best_shift_j, best_angle)
        for start in range(0, len(triples), GRID_BATCH_SIZE):
            batch = triples[start : start + GRID_BATCH_SIZE]
            N = len(batch)
            shift_i_t = torch.tensor([t[0] for t in batch], device=device, dtype=torch.float32).view(N, 1, 1)
            shift_j_t = torch.tensor([t[1] for t in batch], device=device, dtype=torch.float32).view(N, 1, 1)
            angles_rad = torch.tensor([math.radians(-t[2]) for t in batch], device=device, dtype=torch.float32)
            cos_a_t = torch.cos(angles_rad).view(N, 1, 1)
            sin_a_t = torch.sin(angles_rad).view(N, 1, 1)
            warped = apply_transform_batched(g1_filled, shift_i_t, shift_j_t, cos_a_t, sin_a_t)
            errs = discrepancy_batched(g0_filled, warped)
            batch_best_idx = errs.argmin().item()
            batch_best_err = errs[batch_best_idx].item()
            if batch_best_err < best_err:
                best_err = batch_best_err
                best_triple = batch[batch_best_idx]
        best_shift_i, best_shift_j, best_angle = best_triple

        step_shift = step_shift / 5.0 if refine_shift else step_shift
        step_angle = step_angle / 5.0 if refine_angle else step_angle

    return (float(best_shift_i), float(best_shift_j), float(best_angle))

In [ ]:
# Apply register_equal_exposure to two randomly chosen images from each exposure group
def load_grayscale_for_debug(ii):
    with Image.open(ii.path) as img:
        arr = np.array(img).astype(np.float32) / 255.0
    if arr.ndim == 3:
        arr = arr.mean(axis=2)
    return torch.from_numpy(arr).cuda().to(torch.float32)

def apply_transform_single(img, shift_i, shift_j, angle_deg, device):
    H, W = img.shape
    ci, cj = H / 2.0, W / 2.0
    angle_rad = math.radians(-angle_deg)
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)
    ii = torch.arange(H, device=device, dtype=torch.float32).view(-1, 1).expand(H, W)
    jj = torch.arange(W, device=device, dtype=torch.float32).view(1, -1).expand(H, W)
    di = ii - ci - shift_i
    dj = jj - cj - shift_j
    i_src = di * cos_a + dj * sin_a + ci
    j_src = -di * sin_a + dj * cos_a + cj
    j_norm = 2.0 * j_src / (W - 1) - 1.0 if W > 1 else torch.zeros_like(j_src)
    i_norm = 2.0 * i_src / (H - 1) - 1.0 if H > 1 else torch.zeros_like(i_src)
    grid = torch.stack([j_norm, i_norm], dim=-1).unsqueeze(0)
    out = torch.nn.functional.grid_sample(
        img.unsqueeze(0).unsqueeze(0), grid, mode="bilinear", padding_mode="zeros", align_corners=True
    )
    return out.squeeze(0).squeeze(0)

for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    if len(group) < 2:
        print(f"exposure_time={exposure_time:.5f} skip (group size {len(group)} < 2)")
        continue
    ii0, ii1 = random.sample(group, 2)
    shift_i, shift_j, rotation = register_equal_exposure(ii0, ii1)
    print(f"exposure_time={exposure_time:.5f} (n={len(group)}) "
          f"shift_i={shift_i:.4f} shift_j={shift_j:.4f} rotation_deg={rotation:.4f} "
          f"# {ii0.path.name} vs {ii1.path.name}")
    # Debug: two crops side by side (crop = moon center, size 2.4 * moon radius)
    g0 = load_grayscale_for_debug(ii0)
    g1 = load_grayscale_for_debug(ii1)
    g1_aligned = apply_transform_single(g1, shift_i, shift_j, rotation, g0.device)
    moon_i, moon_j, moon_r = ii0.moon[0], ii0.moon[1], ii0.moon[2]
    half = 1.2 * moon_r
    i_lo = max(0, int(moon_i - half))
    i_hi = min(g0.shape[0], int(moon_i + half))
    j_lo = max(0, int(moon_j - half))
    j_hi = min(g0.shape[1], int(moon_j + half))
    crop0 = g0[i_lo:i_hi, j_lo:j_hi].cpu().numpy()
    crop1_aligned = g1_aligned[i_lo:i_hi, j_lo:j_hi].cpu().numpy()
    diff = np.abs(crop0.astype(np.float64) - crop1_aligned.astype(np.float64))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    ax1.imshow(crop0, cmap="gray", vmin=0, vmax=1)
    ax1.set_title("First image (crop)")
    ax1.axis("off")
    ax2.imshow(diff, cmap="gray", vmin=0, vmax=1)
    ax2.set_title("|first - aligned second|")
    ax2.axis("off")
    plt.suptitle(f"exposure_time={exposure_time:.5f}")
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO
# register image within exposure group

In [ ]:
if False:
    # Two-image semitransparent debug: same crop region (from first image), both overlaid
    PATH1 = "/home/slavik/e202602_eclipse/data/img_0148_53656946138_o.jpg"
    PATH2 = "/home/slavik/e202602_eclipse/data/img_0149_53655849382_o.jpg"
    
    img1 = Image.open(PATH1)
    img2 = Image.open(PATH2)
    arr1 = np.array(img1).astype(np.float32) / 255.0
    arr2 = np.array(img2).astype(np.float32) / 255.0
    H, W = arr1.shape[0], arr1.shape[1]
    
    t1 = torch.from_numpy(arr1).cuda()
    t2 = torch.from_numpy(arr2).cuda()
    i1, j1, r1 = find_moon(t1, H / 2, W / 2)
    i2, j2, r2 = find_moon(t2, H / 2, W / 2)
    
    # Crop region from first image (same as in find_moon debug)
    half_crop = int(round(1.1 * r1))
    i_min = int(round(i1 - half_crop))
    i_max = int(round(i1 + half_crop))
    j_min = int(round(j1 - half_crop))
    j_max = int(round(j1 + half_crop))
    # Clamp to image bounds so same region works for both
    i_min = max(0, i_min)
    i_max = min(H - 1, i_max)
    j_min = max(0, j_min)
    j_max = min(W - 1, j_max)

    crop1 = arr1[i_min : i_max + 1, j_min : j_max + 1]
    crop2 = arr2[i_min : i_max + 1, j_min : j_max + 1]
    blended = 0.5 * crop1 + 0.5 * crop2
    
    # Draw same debug as find_moon: green circle at center + 36 border points per moon
    plt.figure(figsize=(12, 12))
    plt.imshow(blended)
    
    # Moon 1 (first image) in crop coords
    c1_j = j1 - j_min
    c1_i = i1 - i_min
    plt.gca().add_patch(plt.Circle((c1_j, c1_i), DEBUG_RADIUS_PX, color="green", fill=True))
    for k in range(36):
        angle = 2 * math.pi * k / 36
        bj = c1_j + r1 * math.cos(angle)
        bi = c1_i + r1 * math.sin(angle)
        plt.plot(bj, bi, "g.", markersize=1)
    
    # Moon 2 (second image) in crop coords
    c2_j = j2 - j_min
    c2_i = i2 - i_min
    plt.gca().add_patch(plt.Circle((c2_j, c2_i), DEBUG_RADIUS_PX, color="green", fill=True))
    for k in range(36):
        angle = 2 * math.pi * k / 36
        bj = c2_j + r2 * math.cos(angle)
        bi = c2_i + r2 * math.sin(angle)
        plt.plot(bj, bi, "g.", markersize=1)
    
    plt.title("Two images overlaid (same crop); green = moon centers and radii")
    plt.axis("off")
    plt.tight_layout()
    plt.show()